# Tutorial 2: ACS 5-Year Aggregate Data

This tutorial covers the most common use case: fetching aggregate statistics from the ACS 5-Year estimates.

**Goal:** Get poverty statistics for all places (cities, towns, CDPs) in a state.

## Setup

In [1]:
import os
from cendat import CenDatHelper
from dotenv import load_dotenv

load_dotenv()
cdh = CenDatHelper(key=os.getenv("CENSUS_API_KEY"))

✅ API key loaded successfully.


## Step 1: Find the ACS 5-Year Product

In [2]:
# The \) at the end matches products ending with a closing paren,
# which filters out sub-products like /profile, /subject, etc.
cdh.list_products(years=[2023], patterns=r"acs/acs5\)")
cdh.set_products()

✅ Product set: 'ACS 5-Year Detailed Tables (2023/acs/acs5)' (Vintage: [2023])


## Step 2: Explore Variable Groups

For products like ACS with thousands of variables, groups are essential:

In [3]:
# Search for poverty-related groups
cdh.list_groups(patterns="poverty")

[{'name': 'B17015',
  'description': 'Poverty Status in the Past 12 Months of Families by Family Type by Social Security Income by Supplemental Security Income (SSI) and Cash Public Assistance Income',
  'product': 'ACS 5-Year Detailed Tables (2023/acs/acs5)',
  'vintage': [2023],
  'url': 'http://api.census.gov/data/2023/acs/acs5'},
 {'name': 'B17016',
  'description': 'Poverty Status in the Past 12 Months of Families by Family Type by Work Experience of Householder and Spouse',
  'product': 'ACS 5-Year Detailed Tables (2023/acs/acs5)',
  'vintage': [2023],
  'url': 'http://api.census.gov/data/2023/acs/acs5'},
 {'name': 'B17017',
  'description': 'Poverty Status in the Past 12 Months by Household Type by Age of Householder',
  'product': 'ACS 5-Year Detailed Tables (2023/acs/acs5)',
  'vintage': [2023],
  'url': 'http://api.census.gov/data/2023/acs/acs5'},
 {'name': 'B17018',
  'description': 'Poverty Status in the Past 12 Months of Families by Household Type by Educational Attainment

In [4]:
# Let's use B17001 (Poverty Status by Sex by Age)
cdh.set_groups("B17001")

# See what variables are in this group
cdh.describe_groups()

✅ Groups set: B17001



--- Group: B17001 (Poverty Status in the Past 12 Months by Sex by Age) ---

  Product: ACS 5-Year Detailed Tables (2023/acs/acs5) (Vintage: 2023)
      B17001_001E: Total:
        B17001_002E: Income in the past 12 months below poverty level:
          B17001_003E: Male:
            B17001_004E: Under 5 years
            B17001_005E: 5 years
            B17001_006E: 6 to 11 years
            B17001_007E: 12 to 14 years
            B17001_008E: 15 years
            B17001_009E: 16 and 17 years
            B17001_010E: 18 to 24 years
            B17001_011E: 25 to 34 years
            B17001_012E: 35 to 44 years
            B17001_013E: 45 to 54 years
            B17001_014E: 55 to 64 years
            B17001_015E: 65 to 74 years
            B17001_016E: 75 years and over
          B17001_017E: Female:
            B17001_018E: Under 5 years
            B17001_019E: 5 years
            B17001_020E: 6 to 11 years
            B17001_021E: 12 to 14 years
            B17001_022E: 15 years
  

## Step 3: Select Variables and Geography

In [5]:
# B17001_001E = Total population for poverty calculation
# B17001_002E = Population below poverty level
cdh.set_variables(["B17001_001E", "B17001_002E"])

# 160 = Places (cities, towns, CDPs)
cdh.set_geos(["160"])

✅ Variables set:
  - Product: ACS 5-Year Detailed Tables (2023/acs/acs5) (Vintage: [2023])
    Variables: B17001_001E, B17001_002E
✅ Geographies set: 'place' (requires `within` for: state)


## Step 4: Get Data with Names

In [6]:
response = cdh.get_data(
    include_names=True,      # Include place names
    include_attributes=True  # Include margins of error
)

✅ Parameters created for 1 geo-variable/group combinations.


✅ Data fetching complete. Stacking results.


## Step 5: Analyze

In [7]:
# Convert to DataFrame
df = response.to_polars(concat=True, destring=True)
df.glimpse()

Rows: 32325
Columns: 15
$ NAME          <str> 'Abanda CDP, Alabama', 'Abbeville city, Alabama', 'Adamsville city, Alabama', 'Addison town, Alabama', 'Akron town, Alabama', 'Alabaster city, Alabama', 'Albertville city, Alabama', 'Alexander City city, Alabama', 'Alexandria CDP, Alabama', 'Aliceville city, Alabama'
$ B17001_001E   <i64> 48, 2306, 4235, 651, 354, 33224, 22180, 14240, 3420, 2338
$ B17001_002E   <i64> 0, 508, 798, 76, 38, 1768, 3795, 3102, 182, 1037
$ B17001_001MA <null> null, null, null, null, null, null, null, null, null, null
$ B17001_002MA <null> null, null, null, null, null, null, null, null, null, null
$ B17001_002EA <null> null, null, null, null, null, null, null, null, null, null
$ B17001_001EA <null> null, null, null, null, null, null, null, null, null, null
$ B17001_001M   <i64> 71, 300, 69, 176, 227, 90, 158, 46, 639, 459
$ B17001_002M   <i64> 13, 170, 435, 53, 40, 465, 994, 554, 139, 369
$ state         <str> '01', '01', '01', '01', '01', '01', '01', '01', '01', 

In [8]:
# Quick tabulation: how many places have >10,000 population?
response.tabulate("state", where="B17001_001E > 10_000")

shape: (52, 5)
┌───────┬─────┬──────┬───────┬────────┐
│ state ┆   n ┆  pct ┆  cumn ┆ cumpct │
╞═══════╪═════╪══════╪═══════╪════════╡
│    01 ┆  70 ┆  1.7 ┆    70 ┆    1.7 │
│    02 ┆   7 ┆  0.2 ┆    77 ┆    1.9 │
│    04 ┆  68 ┆  1.7 ┆   145 ┆    3.5 │
│    05 ┆  35 ┆  0.8 ┆   180 ┆    4.4 │
│    06 ┆ 494 ┆ 12.0 ┆   674 ┆   16.4 │
│    08 ┆  69 ┆  1.7 ┆   743 ┆   18.0 │
│    09 ┆  34 ┆  0.8 ┆   777 ┆   18.9 │
│    10 ┆  11 ┆  0.3 ┆   788 ┆   19.1 │
│    11 ┆   1 ┆  0.0 ┆   789 ┆   19.2 │
│    12 ┆ 342 ┆  8.3 ┆ 1,131 ┆   27.5 │
│    13 ┆ 108 ┆  2.6 ┆ 1,239 ┆   30.1 │
│    15 ┆  33 ┆  0.8 ┆ 1,272 ┆   30.9 │
│    16 ┆  25 ┆  0.6 ┆ 1,297 ┆   31.5 │
│    17 ┆ 223 ┆  5.4 ┆ 1,520 ┆   36.9 │
│    18 ┆  83 ┆  2.0 ┆ 1,603 ┆   38.9 │
│    19 ┆  38 ┆  0.9 ┆ 1,641 ┆   39.8 │
│    20 ┆  34 ┆  0.8 ┆ 1,675 ┆   40.7 │
│    21 ┆  41 ┆  1.0 ┆ 1,716 ┆   41.7 │
│    22 ┆  57 ┆  1.4 ┆ 1,773 ┆   43.0 │
│    23 ┆  12 ┆  0.3 ┆ 1,785 ┆   43.3 │
│    24 ┆ 135 ┆  3.3 ┆ 1,920 ┆   46.6 │
│    25 ┆  89 ┆  2.2 ┆ 2,

In [9]:
# Weighted by population
response.tabulate(
    "state",
    weight_var="B17001_001E",
    where="B17001_001E > 10_000"
)

shape: (52, 5)
┌───────┬────────────┬──────┬─────────────┬────────┐
│ state ┆          n ┆  pct ┆        cumn ┆ cumpct │
╞═══════╪════════════╪══════╪═════════════╪════════╡
│    01 ┆  2,355,550 ┆  1.2 ┆   2,355,550 ┆    1.2 │
│    02 ┆    403,280 ┆  0.2 ┆   2,758,830 ┆    1.4 │
│    04 ┆  6,119,211 ┆  3.0 ┆   8,878,041 ┆    4.4 │
│    05 ┆  1,359,339 ┆  0.7 ┆  10,237,380 ┆    5.1 │
│    06 ┆ 34,506,198 ┆ 17.1 ┆  44,743,578 ┆   22.2 │
│    08 ┆  4,280,170 ┆  2.1 ┆  49,023,748 ┆   24.3 │
│    09 ┆  1,698,368 ┆  0.8 ┆  50,722,116 ┆   25.2 │
│    10 ┆    253,946 ┆  0.1 ┆  50,976,062 ┆   25.3 │
│    11 ┆    647,874 ┆  0.3 ┆  51,623,936 ┆   25.6 │
│    12 ┆ 14,801,907 ┆  7.3 ┆  66,425,843 ┆   33.0 │
│    13 ┆  4,135,032 ┆  2.1 ┆  70,560,875 ┆   35.0 │
│    15 ┆  1,016,957 ┆  0.5 ┆  71,577,832 ┆   35.5 │
│    16 ┆  1,075,551 ┆  0.5 ┆  72,653,383 ┆   36.1 │
│    17 ┆  9,076,740 ┆  4.5 ┆  81,730,123 ┆   40.6 │
│    18 ┆  3,681,396 ┆  1.8 ┆  85,411,519 ┆   42.4 │
│    19 ┆  1,556,593 ┆  0.8 ┆  